# Painel de Análise de Arquitetura - Mag Regional Independente
Este notebook consolida a avaliação da arquitetura **Solarfall** operando estritamente sobre a **Topologia Magnética Regional (Família Mag Regional)** atuando de forma autônoma (End-to-End).

Após a constatação técnica do Efeito Âncora gerado pelo *Meta Model*, a Família Regional foi promovida a estado da arte do projeto. Seus limiares foram endurecidos (maximizando o F2-Score) para que ela atue simultaneamente como detectora e juíza de Falsos Positivos.

A avaliação é realizada **exclusivamente no Ciclo Solar 25 (2020-2024)**, nosso conjunto de Teste Cego.

## 1. Setup & Carregamento de Dados

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
import joblib
import warnings

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model, SpecialistMXModel, SolarFlarePredictionModel
from sklearn.metrics import classification_report, average_precision_score, matthews_corrcoef, f1_score

warnings.filterwarnings('ignore', category=FutureWarning)

load_dotenv()

# --- Paths ---
SLIDED_PATH = os.getenv("SLIDED_PATH")
BASE_PATH_MAG = os.getenv('REGIONAL_MAG_MODELS_PATH')

# --- Load Data ---
print("Carregando dataset regional do Ciclo 25...")
mag_df = pd.read_parquet(os.path.join(SLIDED_PATH, "mag_regional_slided.parquet"))

test_years = [2020, 2021, 2022, 2023, 2024]

In [ ]:
def extract_test_set_regional(df, time_col, region_col):
    """Extrai Teste Cego para Família Regional (por HARP, sem purga)."""
    df = df.sort_values([region_col, time_col]).reset_index(drop=True).copy()
    harp_birth = df.groupby(region_col)[time_col].min().dt.year.to_dict()
    df['harp_birth_year'] = df[region_col].map(harp_birth)
    return df[df['harp_birth_year'].isin(test_years)].copy().reset_index(drop=True)


In [ ]:
test_mag = extract_test_set_regional(mag_df, 'T_REC_round', 'REGION_ID')

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'

X_test_mag = test_mag.drop(columns=[target_class, target_flux, 'T_REC_round', 'REGION_ID', 'DATASET_QUERY', 'harp_birth_year'], errors='ignore')

print(f"Família Regional Autônoma: {len(X_test_mag)} amostras disponíveis no Teste Cego (Ciclo 25)")

## 2. Carregamento dos Modelos End-to-End

In [ ]:
models_mag = {
    'gk': GatekeeperModel.load(os.path.join(BASE_PATH_MAG, 'gatekeeper_v1.joblib')),
    'gf': GreatFilterModel.load(os.path.join(BASE_PATH_MAG, 'great_filter_v1.joblib')),
    's910': Specialist910Model.load(os.path.join(BASE_PATH_MAG, 'specialist_910_v1.joblib')),
    'smx': SpecialistMXModel.load(os.path.join(BASE_PATH_MAG, 'specialist_mx_v1.joblib'))
}

print("Limiares endurecidos embutidos na cascata autônoma:")
print(f"GK: {models_mag['gk'].threshold:.4f} | GF: {models_mag['gf'].threshold:.4f} | S910: {models_mag['s910'].threshold:.4f}")

## 3. Inferência em Cascata & Avaliação de Escoamento
O *Funnel Report* demonstra o corte impiedoso de ruído topológico em cada estágio.

In [ ]:
def analyze_funnel(step_name, y_raw_before, y_raw_after, target_threshold):
    orig_total = len(y_raw_before)
    surv_total = len(y_raw_after)
    if orig_total == 0: return
    
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    orig_pos = (y_raw_before >= target_threshold).sum()
    orig_neg = (y_raw_before < target_threshold).sum()
    surv_pos = (y_raw_after >= target_threshold).sum()
    surv_neg = (y_raw_after < target_threshold).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    display(Markdown(f"**Funnel Report: {step_name}**"))
    print(f"Volume: {orig_total} -> {surv_total} (-{red_pct:.1f}%)")
    print(f"Ruído Eliminado (< Classe Alvo): {noise_reduction:.1f}%")
    print(f"Sinal Retido (>= Classe Alvo): {signal_retention:.1f}%")
    print("-" * 40)

In [ ]:
# --- 1. Gatekeeper ---
y_true_gk = (test_mag[target_class] >= 3).astype(int)
y_pred_gk = models_mag['gk'].predict(X_test_mag)

mask_gf = (y_pred_gk == 1)
analyze_funnel("Entrada -> Gatekeeper", test_mag[target_class], test_mag[target_class][mask_gf], target_threshold=3)

# --- 2. Great Filter ---
X_gf = X_test_mag[mask_gf]
y_true_gf = (test_mag[target_class][mask_gf] >= 3).astype(int)
y_pred_gf = models_mag['gf'].predict(X_gf)

mask_s910 = (y_pred_gf == 1)
y_raw_s910 = test_mag[target_class][mask_gf][mask_s910]
analyze_funnel("Gatekeeper -> Great Filter", test_mag[target_class][mask_gf], y_raw_s910, target_threshold=4)

# --- 3. Specialist 910 (O Juiz das classes M/X) ---
X_s910 = X_gf[mask_s910]
y_true_s910 = (y_raw_s910 >= 4).astype(int)
y_pred_s910 = models_mag['s910'].predict(X_s910)
y_prob_s910 = models_mag['s910'].predict_proba(X_s910)[:, 1]

mask_smx = (y_pred_s910 == 1)
y_raw_smx = y_raw_s910[mask_smx]
analyze_funnel("Great Filter -> Specialist 910", y_raw_s910, y_raw_smx, target_threshold=5)

# --- 4. Specialist MX (Regressão Final) ---
X_smx = X_s910[mask_smx]
y_true_smx = (y_raw_smx >= 5).astype(int)
y_pred_cont = models_mag['smx'].predict(X_smx)
y_pred_smx = (y_pred_cont >= -4.0).astype(int)

## 4. Métricas de Estado da Arte (Specialist 910 Autônomo)
Sendo a camada final de classificação binária severa (separando M/X de C), extraímos do *Specialist 910* as métricas oficiais para a sua Tabela de Comparação do artigo.

In [ ]:
tss_fam2 = SolarFlarePredictionModel.calculate_tss(y_true_s910, y_pred_s910)
hss_fam2 = SolarFlarePredictionModel.calculate_hss(y_true_s910, y_pred_s910)
f1_fam2 = f1_score(y_true_s910, y_pred_s910)
pr_auc_s910 = average_precision_score(y_true_s910, y_prob_s910)

print("Resultados do Specialist 910 (Preditor Final Autônomo):")
print(f"  TSS:    {tss_fam2:.3f}")
print(f"  HSS:    {hss_fam2:.3f}")
print(f"  F1:     {f1_fam2:.3f}")
print(f"  PR-AUC: {pr_auc_s910:.3f}")

print("\nClassification Report (S910):")
print(models_mag['s910'].get_classification_report(y_true_s910, y_pred_s910, ['< M', 'M/X']))

## 5. Visuais Científicos para o Relatório Final

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve

# ---------------------------------------------------------
# 5.1 DIAGRAMA DE ESCOAMENTO (FUNNEL REPORT)
# ---------------------------------------------------------
volumes_mag = [
    len(X_test_mag),
    len(X_gf),
    len(X_s910),
    len(X_smx)
]

etapas = ["Entrada (Pool)", "Sobreviventes do Gatekeeper", "Sobreviventes do Great Filter", "Sobreviventes do S910"]

fig_funnel = go.Figure(go.Funnel(
    name = 'Família Regional Mag (Autônoma)',
    y = etapas,
    x = volumes_mag,
    textinfo = "value+percent initial",
    marker = {"color": "#636efa"}
))

fig_funnel.update_layout(
    title="Escoamento da Cascata Regional: Supressão de Ruído Magnético",
    yaxis_title="Estágios de Triagem"
)
fig_funnel.show()

# ---------------------------------------------------------
# 5.2 HEATMAP DE ERROS DO SPECIALIST 910
# ---------------------------------------------------------
err_s910 = models_mag['s910'].analyze_error_distribution(y_true_s910, y_pred_s910, test_mag[target_flux][mask_gf][mask_s910])

plt.figure(figsize=(8, 6))
matriz_erros = err_s910[['FN (Miss)', 'FP (False Alarm)']].astype(float)

sns.heatmap(matriz_erros, annot=True, fmt="g", cmap="Blues", cbar=False,
            annot_kws={"size": 12, "weight": "bold"}, linewidths=.5)
plt.title("Distribuição Física de Erros (Specialist 910 Autônomo)", fontsize=14, pad=10)
plt.ylabel("Classe Solar Relevante")
plt.xlabel("Tipo de Erro")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5.3 CURVA DE PRECISÃO-RECALL (PR-AUC) DA REDE AUTÔNOMA
# ---------------------------------------------------------
# Como a rede não depende de um oráculo externo, sua competência de alerta é avaliada
# diretamente pelas probabilidades finais calculadas na barreira do M/X.
precision_II, recall_II, _ = precision_recall_curve(y_true_s910, y_prob_s910)

plt.figure(figsize=(9, 6))
plt.plot(recall_II, precision_II, label=f'Mag Regional End-to-End - AUC: {pr_auc_s910:.3f}', color='#636efa', linewidth=2.5)

plt.title('Curva Precisão-Recall Autônoma: Previsão Severa M/X', fontsize=15, pad=15)
plt.xlabel('Recall (Sensibilidade)', fontsize=12)
plt.ylabel('Precisão (VPP)', fontsize=12)
plt.legend(loc="upper right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5.4 FEATURE IMPORTANCE DO SPECIALIST 910
# ---------------------------------------------------------
print("\n--- A FÍSICA DO ALERTA M/X ---")
importance_df = models_mag['s910'].get_feature_importance().head(10)
importance_df_plot = importance_df.sort_values(by='importance_gain', ascending=True)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=importance_df_plot,
    x='importance_gain',
    y='feature',
    palette="viridis"
)

for p in ax.patches:
    ax.annotate(f"{p.get_width():.3f}",
                (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center',
                xytext=(5, 0), textcoords='offset points',
                fontsize=10, fontweight='bold')

plt.title('Importância das Variáveis Topológicas (Specialist 910)', fontsize=15, pad=15)
plt.xlabel('Normalized Information Gain', fontsize=12)
plt.ylabel('Parâmetros SHARP Intensivos', fontsize=12)
plt.tight_layout()
plt.show()